In [ ]:
from pathlib import Path
import os

# Set PROJECT_DATA_DIR before launching Jupyter to use data stored elsewhere.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".gitignore").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = Path(os.environ.get("PROJECT_DATA_DIR", str(PROJECT_ROOT / "data"))).expanduser().resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, average_precision_score


In [ ]:
script_dir = os.getcwd()

csv_path = os.path.join(script_dir, str(DATA_DIR / 'Fraud Detection Transactions Dataset.csv'))

df = pd.read_csv(csv_path)

In [ ]:
df.head(5)

In [ ]:
features = ['Transaction_ID', 'User_ID', 'Transaction_Amount', 'Transaction_Type', 'Timestamp', 'Account_Balance', 'Device_Type', 'Location', 'Merchant_Category', 'IP_Address_Flag', 'Daily_Transaction_Count', 'Avg_Transaction_Amount_7d', 'Failed_Transaction_Count_7d', 'Card_Type', 'Card_Age', 'Transaction_Distance', 'Authentication_Method', 'Risk_Score', 'Is_Weekend']
target = ['Fraud_Label']

In [ ]:
print(df[features].head())

In [ ]:
# Identify categorical columns that need one-hot encoding
categorical_columns = ['Transaction_Type', 'Device_Type', 'Location', 'Merchant_Category', 'Card_Type', 'Authentication_Method']

# Apply one-hot encoding
df_encoded = pd.get_dummies(df, columns=categorical_columns, drop_first=False)

print("Original shape:", df.shape)
print("Encoded shape:", df_encoded.shape)
print("\nNew columns created:")
print(df_encoded.columns.tolist())

In [ ]:
# Drop leakage + unnecessary columns
columns_to_drop = [
    'Transaction_ID',
    'User_ID',
    'Timestamp',
    'Risk_Score'   # if removing leakage
]

X = df_encoded.drop(columns=columns_to_drop + ['Fraud_Label'])
y = df_encoded['Fraud_Label']


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

clf = Pipeline([
    ("xgb", XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=0,
        n_jobs=-1,
        eval_metric="logloss"  # avoids warning, good default for binary
    ))
])

clf.fit(X_train, y_train)


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, average_precision_score 
y_pred = clf.predict(X_test) 
y_proba = clf.predict_proba(X_test)[:, 1] 
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred)) 
print("\nReport:\n", classification_report(y_test, y_pred)) 
print("ROC-AUC:", roc_auc_score(y_test, y_proba)) 
print("PR-AUC :", average_precision_score(y_test, y_proba))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=[0, 1],   # or your real class names
    values_format='d'
)
plt.show()
